In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_pipeline.bronze;

CREATE SCHEMA IF NOT EXISTS my_pipeline.silver;

CREATE SCHEMA IF NOT EXISTS my_pipeline.gold;

In [0]:
%sql
SHOW SCHEMAS IN my_pipeline;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
url ="https://archive-api.open-meteo.com/v1/archive?latitude=13.0827&longitude=80.2707&start_date=2026-07-01&end_date=2026-07-07&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code,rain,apparent_temperature,precipitation,surface_pressure,cloud_cover,wind_direction_10m,wind_gusts_10m"

In [0]:
import requests
response = requests.get(url)
data=response.json()
print(data)

{'latitude': 13.110721, 'longitude': 80.2459, 'generationtime_ms': 0.21314620971679688, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 12.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h', 'weather_code': 'wmo code', 'rain': 'mm', 'apparent_temperature': '°C', 'precipitation': 'mm', 'surface_pressure': 'hPa', 'cloud_cover': '%', 'wind_direction_10m': '°', 'wind_gusts_10m': 'km/h'}, 'hourly': {'time': ['2026-07-01T00:00', '2026-07-01T01:00', '2026-07-01T02:00', '2026-07-01T03:00', '2026-07-01T04:00', '2026-07-01T05:00', '2026-07-01T06:00', '2026-07-01T07:00', '2026-07-01T08:00', '2026-07-01T09:00', '2026-07-01T10:00', '2026-07-01T11:00', '2026-07-01T12:00', '2026-07-01T13:00', '2026-07-01T14:00', '2026-07-01T15:00', '2026-07-01T16:00', '2026-07-01T17:00', '2026-07-01T18:00', '2026-07-01T19:00', '2026-07-01T20:00', '2026-07-01T21:00', '2026-07-01T22:00', '2026-07-01T23:00', '2026-

In [0]:
hourly=data["hourly"]
print(hourly.keys())

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'weather_code', 'rain', 'apparent_temperature', 'precipitation', 'surface_pressure', 'cloud_cover', 'wind_direction_10m', 'wind_gusts_10m'])


In [0]:
rows = list(zip(
    hourly["time"],
    hourly["temperature_2m"],
    hourly["relative_humidity_2m"],
    hourly["apparent_temperature"],
    hourly["precipitation"],
    hourly["rain"],
    hourly["weather_code"],
    hourly["surface_pressure"],
    hourly["cloud_cover"],
    hourly["wind_speed_10m"],
    hourly["wind_direction_10m"],
    hourly["wind_gusts_10m"]
))

columns = [
    "time",
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "weather_code",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
]

df = spark.createDataFrame(rows, columns)

display(df)

time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
2026-07-01T00:00,28.7,64,31.0,0.0,0.0,3,1003.8,99,14.5,259,34.6
2026-07-01T01:00,28.7,64,31.1,0.0,0.0,3,1004.5,100,14.6,257,30.6
2026-07-01T02:00,29.4,63,32.1,0.0,0.0,3,1005.0,99,13.6,255,30.6
2026-07-01T03:00,30.5,60,33.1,0.0,0.0,3,1005.2,98,15.3,253,34.9
2026-07-01T04:00,31.6,56,33.8,0.0,0.0,3,1005.3,99,18.1,257,42.1
2026-07-01T05:00,32.2,55,34.7,0.0,0.0,3,1004.9,100,19.2,251,45.7
2026-07-01T06:00,32.9,53,35.1,0.0,0.0,3,1004.5,100,19.9,261,45.7
2026-07-01T07:00,32.9,54,35.8,0.0,0.0,3,1004.1,100,19.6,252,49.3
2026-07-01T08:00,33.5,54,36.9,0.0,0.0,3,1003.1,82,17.0,247,46.1
2026-07-01T09:00,35.5,47,39.2,0.0,0.0,3,1001.9,98,14.4,269,47.5


In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    df
    .withColumn("location", lit("Chennai"))
    .withColumn("latitude", lit(13.0827))
    .withColumn("longitude", lit(80.2707))
    .withColumn("source", lit("Open-Meteo Historical Weather API"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(bronze_df)

time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,location,latitude,longitude,source,ingestion_timestamp
2026-07-01T00:00,28.7,64,31.0,0.0,0.0,3,1003.8,99,14.5,259,34.6,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T01:00,28.7,64,31.1,0.0,0.0,3,1004.5,100,14.6,257,30.6,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T02:00,29.4,63,32.1,0.0,0.0,3,1005.0,99,13.6,255,30.6,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T03:00,30.5,60,33.1,0.0,0.0,3,1005.2,98,15.3,253,34.9,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T04:00,31.6,56,33.8,0.0,0.0,3,1005.3,99,18.1,257,42.1,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T05:00,32.2,55,34.7,0.0,0.0,3,1004.9,100,19.2,251,45.7,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T06:00,32.9,53,35.1,0.0,0.0,3,1004.5,100,19.9,261,45.7,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T07:00,32.9,54,35.8,0.0,0.0,3,1004.1,100,19.6,252,49.3,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T08:00,33.5,54,36.9,0.0,0.0,3,1003.1,82,17.0,247,46.1,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z
2026-07-01T09:00,35.5,47,39.2,0.0,0.0,3,1001.9,98,14.4,269,47.5,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-29T15:05:52.336Z


In [0]:
print("rows: ",bronze_df.count())
print("columns: ",len(bronze_df.columns))

rows:  4344
columns:  17


In [0]:
bronze_df.write.format("delta").mode("append").saveAsTable("my_pipeline.bronze.weather")

In [0]:
#bronze_check=spark.table('my_pipeline.bronze.weather')
#print("Rows: ",bronze_check.count())
#print("Columns: ",len(bronze_check.columns))

##SILVER

In [0]:
silver_df=spark.table("my_pipeline.bronze.weather")
silver_df.display()

time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,location,latitude,longitude,source,ingestion_timestamp
2026-05-25T19:00,32.3,65,37.5,0.0,0.0,3,1005.4,84,10.6,183,23.4,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-25T20:00,31.5,72,36.9,0.0,0.0,2,1004.9,72,11.8,191,25.6,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-25T21:00,30.9,73,36.1,0.0,0.0,1,1004.3,34,13.1,192,28.1,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-25T22:00,30.4,74,35.5,0.0,0.0,2,1004.4,74,12.7,203,28.4,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-25T23:00,30.1,74,34.8,0.0,0.0,2,1004.7,73,13.1,210,27.7,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-26T00:00,29.9,77,35.3,0.0,0.0,3,1004.6,99,10.7,211,28.1,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-26T01:00,30.1,75,35.6,0.0,0.0,3,1005.3,99,8.9,222,23.4,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-26T02:00,31.7,67,36.8,0.0,0.0,1,1005.8,23,9.7,231,24.8,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-26T03:00,34.2,52,37.6,0.0,0.0,0,1006.0,10,13.9,256,34.9,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z
2026-05-26T04:00,36.1,46,39.7,0.0,0.0,2,1006.3,76,12.3,266,35.6,Chennai,13.0827,80.2707,Open-Meteo Historical Weather API,2026-09-26T16:07:47.230Z


In [0]:
from pyspark.sql.functions import *
silver_df.select([sum(when(col(c).isNull() ,1).otherwise(0)).alias(c) for c in silver_df.columns]).show()

+----+--------------+--------------------+--------------------+-------------+----+------------+----------------+-----------+--------------+------------------+--------------+--------+--------+---------+------+-------------------+
|time|temperature_2m|relative_humidity_2m|apparent_temperature|precipitation|rain|weather_code|surface_pressure|cloud_cover|wind_speed_10m|wind_direction_10m|wind_gusts_10m|location|latitude|longitude|source|ingestion_timestamp|
+----+--------------+--------------------+--------------------+-------------+----+------------+----------------+-----------+--------------+------------------+--------------+--------+--------+---------+------+-------------------+
|   0|             0|                   0|                   0|            0|   0|           0|               0|          0|             0|                 0|             0|       0|       0|        0|     0|                  0|
+----+--------------+--------------------+--------------------+-------------+----+--

In [0]:
total_rows=silver_df.count()
unique_rows=silver_df.dropDuplicates().count() 
print("Total rows: ",total_rows)
print("Unique rows: ",unique_rows)


Total rows:  4344
Unique rows:  4344


In [0]:
print("Total timestamps:", silver_df.select("time").count())
print("Unique timestamps:", silver_df.select("time").distinct().count())

Total timestamps: 4344
Unique timestamps: 4344


In [0]:
silver_df.printSchema()

root
 |-- time: string (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
silver_df=silver_df.withColumn("time",to_timestamp("time"))
silver_df.printSchema()

root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
silver_df.select("time").limit(10).display()

time
2026-04-19T14:00:00.000Z
2026-04-19T15:00:00.000Z
2026-04-19T16:00:00.000Z
2026-04-19T17:00:00.000Z
2026-04-19T18:00:00.000Z
2026-04-19T19:00:00.000Z
2026-04-19T20:00:00.000Z
2026-04-19T21:00:00.000Z
2026-04-19T22:00:00.000Z
2026-04-19T23:00:00.000Z


In [0]:
silver_df=silver_df.withColumn("date",to_date("time"))
silver_df.select("date").display()

date
2026-05-25
2026-05-25
2026-05-25
2026-05-25
2026-05-25
2026-05-26
2026-05-26
2026-05-26
2026-05-26
2026-05-26


In [0]:
silver_df.columns

['time',
 'temperature_2m',
 'relative_humidity_2m',
 'apparent_temperature',
 'precipitation',
 'rain',
 'weather_code',
 'surface_pressure',
 'cloud_cover',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'location',
 'latitude',
 'longitude',
 'source',
 'ingestion_timestamp',
 'date']

In [0]:
silver_df=silver_df.drop("location","latitude","longitude","source","ingestion_timestamp")
silver_df.columns

['time',
 'temperature_2m',
 'relative_humidity_2m',
 'apparent_temperature',
 'precipitation',
 'rain',
 'weather_code',
 'surface_pressure',
 'cloud_cover',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'date']

In [0]:
silver_df.summary().display()

summary,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
count,4344,4344,4344,4344,4344,4344,4344,4344,4344,4344,4344
mean,28.904005524861887,70.09530386740332,32.798112338858196,0.057458563535911604,0.057458563535911604,6.762200736648251,1008.3779235727438,50.53430018416206,11.310405156537751,141.41160220994476,28.610405156537745
stddev,3.738989063024436,14.562042579565711,4.565906061542634,0.37897128669153907,0.37897128669153907,15.747499268674103,3.9513199708455122,39.15596640494722,4.255237719720207,83.39442698711628,10.54858554865763
min,19.2,30,20.6,0.0,0.0,0,999.4,0,0.0,1,1.8
25%,26.1,59,29.2,0.0,0.0,0,1005.3,12,8.6,56,21.2
50%,28.7,72,33.2,0.0,0.0,2,1007.8,45,11.4,149,28.1
75%,31.4,82,35.9,0.0,0.0,3,1011.6,95,14.1,201,36.4
max,40.2,99,47.3,12.1,12.1,65,1018.3,100,28.3,360,65.9


In [0]:
#dropduplicates
silver_df=silver_df.dropDuplicates()
print("rows after removig duplicates",silver_df.count())

rows after removig duplicates 4344


In [0]:
#arrange cols in order
silver_df = silver_df.select(
    "time",
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
)

display(silver_df)

time,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
2026-05-26T15:00:00.000Z,33.6,37.5,51,0.0,0.0,1,1006.0,34,6.7,232,18.7
2026-05-28T07:00:00.000Z,39.2,43.7,35,0.0,0.0,0,1005.7,6,13.1,288,38.5
2026-05-28T08:00:00.000Z,39.9,44.6,33,0.0,0.0,0,1004.9,15,9.6,311,37.4
2026-05-28T12:00:00.000Z,34.1,39.0,60,0.0,0.0,2,1003.9,71,13.4,133,34.9
2026-05-28T19:00:00.000Z,31.3,37.5,76,0.0,0.0,3,1005.9,99,10.7,209,27.4
2026-05-30T11:00:00.000Z,33.3,38.8,60,0.0,0.0,3,1004.1,84,5.4,90,20.2
2026-06-01T08:00:00.000Z,34.4,39.6,56,0.2,0.2,51,1006.6,42,14.0,66,46.4
2026-06-01T19:00:00.000Z,30.4,36.3,77,0.0,0.0,3,1006.1,100,9.9,201,24.1
2026-06-02T03:00:00.000Z,32.5,38.2,67,0.0,0.0,1,1007.5,24,10.3,247,28.4
2026-06-02T06:00:00.000Z,37.5,42.3,40,0.3,0.3,51,1006.3,8,9.3,279,27.7


In [0]:
print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))
print(silver_df.columns)

Rows: 4344
Columns: 12
['time', 'temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'precipitation', 'rain', 'weather_code', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m']


In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("my_pipeline.silver.weather")

In [0]:
silver_check=spark.table("my_pipeline.silver.weather")
print("rows: ",silver_check.count())
print("columns: ",len(silver_check.columns))

rows:  4344
columns:  12


##GOLD

In [0]:
gold_df=spark.table("my_pipeline.silver.weather")
gold_df.display()

time,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
2026-05-26T15:00:00.000Z,33.6,37.5,51,0.0,0.0,1,1006.0,34,6.7,232,18.7
2026-05-28T07:00:00.000Z,39.2,43.7,35,0.0,0.0,0,1005.7,6,13.1,288,38.5
2026-05-28T08:00:00.000Z,39.9,44.6,33,0.0,0.0,0,1004.9,15,9.6,311,37.4
2026-05-28T12:00:00.000Z,34.1,39.0,60,0.0,0.0,2,1003.9,71,13.4,133,34.9
2026-05-28T19:00:00.000Z,31.3,37.5,76,0.0,0.0,3,1005.9,99,10.7,209,27.4
2026-05-30T11:00:00.000Z,33.3,38.8,60,0.0,0.0,3,1004.1,84,5.4,90,20.2
2026-06-01T08:00:00.000Z,34.4,39.6,56,0.2,0.2,51,1006.6,42,14.0,66,46.4
2026-06-01T19:00:00.000Z,30.4,36.3,77,0.0,0.0,3,1006.1,100,9.9,201,24.1
2026-06-02T03:00:00.000Z,32.5,38.2,67,0.0,0.0,1,1007.5,24,10.3,247,28.4
2026-06-02T06:00:00.000Z,37.5,42.3,40,0.3,0.3,51,1006.3,8,9.3,279,27.7


In [0]:
from pyspark.sql.functions import to_timestamp, to_date

silver_df = spark.table("my_pipeline.bronze.weather")

silver_df = (
    silver_df
    .withColumn("time", to_timestamp("time"))
    .withColumn("date", to_date("time"))
)

In [0]:
gold_df=silver_df.groupBy("date").agg(
    avg("temperature_2m").alias("avg_temperature"),
    min("temperature_2m").alias("min_temperature"),
    max("temperature_2m").alias("max_temperature"),
    avg("apparent_temperature").alias("avg_apparente_temperature"),
    avg("relative_humidity_2m").alias("avg_relative_humidity"),
    sum("precipitation").alias("total_precipation"),
    sum("rain").alias("total_rain"),
    max("wind_speed_10m").alias("max_wind_speed_10m"),
    max("wind_gusts_10m").alias("max_wind_guests_10m"),
    avg("cloud_cover").alias("avg_cloud_cover")
)

In [0]:
gold_df.columns

['date',
 'avg_temperature',
 'min_temperature',
 'max_temperature',
 'avg_apparente_temperature',
 'avg_relative_humidity',
 'total_precipation',
 'total_rain',
 'max_wind_speed_10m',
 'max_wind_guests_10m',
 'avg_cloud_cover']

In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("my_pipeline.gold.weather")

In [0]:
gold_check = spark.table("my_pipeline.gold.weather")

print("Rows:", gold_check.count())
print("Columns:", len(gold_check.columns))

display(gold_check)

Rows: 181
Columns: 11


date,avg_temperature,min_temperature,max_temperature,avg_apparente_temperature,avg_relative_humidity,total_precipation,total_rain,max_wind_speed_10m,max_wind_guests_10m,avg_cloud_cover
2026-06-08,33.925000000000004,29.8,37.9,37.11666666666668,50.333333333333336,0.30000000000000004,0.30000000000000004,19.5,44.6,89.625
2026-06-11,30.387500000000003,28.1,32.8,35.225,70.58333333333333,4.6000000000000005,4.6000000000000005,15.2,36.0,96.58333333333333
2026-06-28,29.474999999999994,25.4,34.6,35.400000000000006,80.29166666666667,18.599999999999998,18.599999999999998,15.7,40.7,59.291666666666664
2026-05-25,34.04583333333333,30.1,39.0,38.70416666666666,56.583333333333336,0.0,0.0,15.9,43.9,32.5
2026-05-26,33.74583333333333,29.6,39.5,37.87083333333334,53.208333333333336,0.7,0.7,16.5,43.2,50.666666666666664
2026-06-01,32.38333333333333,29.1,38.9,36.99999999999999,61.458333333333336,1.0,1.0,17.1,46.4,59.041666666666664
2026-06-15,32.16250000000001,29.2,37.2,36.75833333333334,59.958333333333336,1.7,1.7,14.2,37.4,53.416666666666664
2026-06-30,31.470833333333335,28.7,36.3,35.2125,59.708333333333336,2.8,2.8,18.1,40.0,93.83333333333333
2026-05-30,31.91250000000001,29.1,36.6,36.654166666666676,63.0,0.8999999999999999,0.8999999999999999,21.0,48.2,92.625
2026-06-04,33.525,30.7,38.5,37.3125,54.666666666666664,2.9000000000000004,2.9000000000000004,17.6,41.8,63.208333333333336
